# Global AI Regulation Divergence Index
## Analysing regulatory divergence across 30 countries · April 2026
**Author:** Angkuran Dey · Google Data Analytics Professional Certificate · MSc, University of Bristol

This notebook analyses how 30 countries across 6 regions approach AI governance, 
scoring each against 8 parameters on a 0–2 scale (0 = absent, 1 = voluntary/partial, 
2 = binding/enforced). The result is a Divergence Index revealing which countries 
are Regulatory Hawks, Developing Frameworks, or Regulatory Doves.

### Tools used
- **Python** (pandas, plotly) — analysis and choropleth visualisation
- **Tableau Public** — interactive dashboard
- **GitHub** — version control and documentation

In [3]:
import pandas as pd
import plotly.express as px
print("Libraries loaded successfully")
print("Project: Global AI Regulation Divergence Index")
print("Author: Angkuran Dey")

Libraries loaded successfully
Project: Global AI Regulation Divergence Index
Author: Angkuran Dey


## 1. Data Loading
The dataset was built through primary research across government sources, 
legislative trackers (IAPP, mindfoundry.ai), and academic publications 
(Royal Society Open Science, Feb 2026). Each country was scored manually 
against 8 governance parameters. Data collected April 2026.


In [4]:
# Load the dataset
df = pd.read_excel("Global_AI_Regulation_Divergence_Index.xlsx", sheet_name="Country Data", header=1)

# Preview
print(f"Dataset loaded: {df.shape[0]} countries, {df.shape[1]} columns")
print(df[["country", "region", "regulation_score", "regulatory_cluster"]].to_string())

Dataset loaded: 30 countries, 17 columns
           country        region  regulation_score    regulatory_cluster
0   European Union        Europe                16       Regulatory Hawk
1   United Kingdom        Europe                 5       Regulatory Dove
2          Germany        Europe                16       Regulatory Hawk
3           France        Europe                16       Regulatory Hawk
4      Switzerland        Europe                 7  Developing Framework
5           Norway        Europe                16       Regulatory Hawk
6           Sweden        Europe                16       Regulatory Hawk
7      Netherlands        Europe                16       Regulatory Hawk
8            Italy        Europe                16       Regulatory Hawk
9            Spain        Europe                16       Regulatory Hawk
10   United States      Americas                 4       Regulatory Dove
11          Canada      Americas                 7  Developing Framework
12        

## 2. Regional Analysis
Grouping countries by region to identify which parts of the world 
are leading and lagging on AI governance. Average score, range, 
and country count shown per region.

In [5]:
# Regional analysis
regional = df.groupby("region")["regulation_score"].agg(["mean","min","max","count"]).round(2)
regional.columns = ["avg_score", "min_score", "max_score", "num_countries"]
regional = regional.sort_values("avg_score", ascending=False)
print("=== REGIONAL AVERAGE SCORES (out of 16) ===")
print(regional.to_string())

=== REGIONAL AVERAGE SCORES (out of 16) ===
              avg_score  min_score  max_score  num_countries
region                                                      
Europe            14.00          5         16             10
Asia-Pacific       9.50          3         16              8
Other              8.00          8          8              1
Americas           5.40          1         13              5
Middle East        3.00          2          4              2
Africa             2.25          1          3              4


## 3. Parameter Adoption Rates
Measuring how widely each of the 8 governance parameters is adopted 
across all 30 countries. This reveals the global enforcement gap — 
the difference between countries that have *written* AI strategies 
and those that have *built* mechanisms to enforce them.

In [6]:
# Parameter adoption rates
params = ["has_binding_legislation","has_risk_based_approach",
          "has_enforcement_mech","has_transparency_oblig",
          "has_human_oversight","has_penalty_framework",
          "has_national_ai_strategy","has_prohibited_practices"]

print("=== PARAMETER ADOPTION (avg score out of 2) ===")
adoption = df[params].mean().round(2).sort_values()
for param, val in adoption.items():
    pct = round(val/2*100, 1)
    bar = "█" * int(pct/5)
    print(f"{param:<32} {val:.2f}  {pct}%  {bar}")

=== PARAMETER ADOPTION (avg score out of 2) ===
has_enforcement_mech             0.77  38.5%  ███████
has_penalty_framework            0.77  38.5%  ███████
has_prohibited_practices         0.97  48.5%  █████████
has_binding_legislation          1.03  51.5%  ██████████
has_human_oversight              1.03  51.5%  ██████████
has_risk_based_approach          1.03  51.5%  ██████████
has_transparency_oblig           1.40  70.0%  ██████████████
has_national_ai_strategy         1.87  93.5%  ██████████████████


## 4. Choropleth World Map
Interactive world map colour-coded by regulation score — deep red (Regulatory Dove) 
through gold to dark green (Regulatory Hawk). Built with Plotly Express using ISO 
alpha-3 country codes against a dark ocean background for maximum visual contrast.

Design choices:
- **Dark background** (#0D1B2A) — makes the colour gradient dramatically more readable
- **7-stop colour scale** — finer gradation between Hawks, Developing Frameworks and Doves
- **Country borders** at 0.5px — visible without overpowering the colour signal
- **Natural Earth projection** — the most geographically accurate world view

Saved as a standalone interactive HTML file — hover over any country to see 
its score, cluster, region and percentage. No Python required to view it.

In [7]:
# Enhanced choropleth map
fig = px.choropleth(
    df,
    locations="iso_alpha3",
    color="regulation_score",
    hover_name="country",
    hover_data={
        "iso_alpha3": False,
        "regulation_score": True,
        "regulatory_cluster": True,
        "region": True,
        "score_pct": True
    },
    color_continuous_scale=[
        [0.0,  "#7B1111"],
        [0.2,  "#C0392B"],
        [0.4,  "#E8956D"],
        [0.55, "#F5DEB3"],
        [0.7,  "#90C990"],
        [0.85, "#2E8B57"],
        [1.0,  "#1A5C2A"]
    ],
    range_color=[0, 16],
    labels={
        "regulation_score": "Score (out of 16)",
        "regulatory_cluster": "Cluster",
        "region": "Region",
        "score_pct": "Score %"
    }
)

fig.update_traces(
    marker_line_color="#2C2C2C",
    marker_line_width=0.5
)

fig.update_geos(
    showframe=False,
    showcoastlines=True,
    coastlinecolor="#888888",
    coastlinewidth=0.5,
    showland=True,
    landcolor="#1C1C1C",
    showocean=True,
    oceancolor="#0D1B2A",
    showlakes=True,
    lakecolor="#0D1B2A",
    showcountries=False,
    bgcolor="#0D1B2A",
    projection_type="natural earth"
)

fig.update_layout(
    title=dict(
        text="<b>Global AI Regulation Divergence Index</b><br><sup>30 Countries · 8 Parameters · April 2026 · Angkuran Dey</sup>",
        x=0.5,
        xanchor="center",
        font=dict(size=20, color="#FFFFFF", family="Arial")
    ),
    paper_bgcolor="#0D1B2A",
    plot_bgcolor="#0D1B2A",
    margin={"r": 20, "t": 80, "l": 20, "b": 20},
    coloraxis_colorbar=dict(
        title=dict(text="Score<br>(out of 16)", font=dict(color="#CCCCCC", size=11)),
        tickfont=dict(color="#CCCCCC"),
        bgcolor="#1C2C3C",
        bordercolor="#444444",
        borderwidth=1,
        thickness=15,
        len=0.6
    )
)

fig.write_html("global_ai_regulation_map.html")
print("Enhanced map saved — open global_ai_regulation_map.html in your browser")

Enhanced map saved — open global_ai_regulation_map.html in your browser


## 5. Top and Bottom Performers
Identifying the 5 highest and 5 lowest scoring countries, 
and the overall cluster breakdown across all 30 nations.

In [8]:
# Top and bottom 5 countries
print("=== TOP 5 REGULATORY HAWKS ===")
print(df[["country","region","regulation_score","regulatory_cluster"]].nlargest(5, "regulation_score").to_string(index=False))

print("\n=== BOTTOM 5 REGULATORY DOVES ===")
print(df[["country","region","regulation_score","regulatory_cluster"]].nsmallest(5, "regulation_score").to_string(index=False))

print("\n=== CLUSTER BREAKDOWN ===")
print(df["regulatory_cluster"].value_counts().to_string())

=== TOP 5 REGULATORY HAWKS ===
       country region  regulation_score regulatory_cluster
European Union Europe                16    Regulatory Hawk
       Germany Europe                16    Regulatory Hawk
        France Europe                16    Regulatory Hawk
        Norway Europe                16    Regulatory Hawk
        Sweden Europe                16    Regulatory Hawk

=== BOTTOM 5 REGULATORY DOVES ===
     country      region  regulation_score regulatory_cluster
      Mexico    Americas                 1    Regulatory Dove
     Nigeria      Africa                 1    Regulatory Dove
   Argentina    Americas                 2    Regulatory Dove
Saudi Arabia Middle East                 2    Regulatory Dove
South Africa      Africa                 2    Regulatory Dove

=== CLUSTER BREAKDOWN ===
regulatory_cluster
Regulatory Hawk         13
Regulatory Dove         13
Developing Framework     4


## 6. The Enforcement Gap — Key Finding
Countries that have published a national AI strategy (score 2 on 
has_national_ai_strategy) but have NO dedicated enforcement mechanism 
(score 0 on has_enforcement_mech). These are nations making promises 
without building the institutional capacity to keep them.

In [9]:
# The enforcement gap — countries with strategy but no enforcement
enforcement_gap = df[(df["has_national_ai_strategy"] == 2) & (df["has_enforcement_mech"] == 0)]
print("=== THE ENFORCEMENT GAP ===")
print("Countries with a national AI strategy but NO enforcement mechanism:")
print(enforcement_gap[["country","region","regulation_score"]].to_string(index=False))
print(f"\nTotal: {len(enforcement_gap)} out of 30 countries")

=== THE ENFORCEMENT GAP ===
Countries with a national AI strategy but NO enforcement mechanism:
       country       region  regulation_score
United Kingdom       Europe                 5
   Switzerland       Europe                 7
 United States     Americas                 4
        Canada     Americas                 7
         Japan Asia-Pacific                 6
     Singapore Asia-Pacific                 7
         India Asia-Pacific                 3
     Australia Asia-Pacific                 5
           UAE  Middle East                 4
  Saudi Arabia  Middle East                 2
         Kenya       Africa                 3
        Rwanda       Africa                 3
        Israel        Other                 8

Total: 13 out of 30 countries


## Summary of Key Findings

| Finding | Detail |
|---|---|
| Global split | 13 Regulatory Hawks · 13 Regulatory Doves · 4 Developing Frameworks |
| Strongest region | Europe — average 14.0/16, driven by the EU AI Act |
| Weakest region | Africa — average 2.25/16, strategy-heavy but law-light |
| Biggest intra-region gap | Asia-Pacific — range of 3 to 16 within a single region |
| Most adopted parameter | National AI strategy — 93.5% adoption |
| Least adopted parameter | Enforcement mechanism — 38.5% adoption |
| Enforcement gap | 13 of 30 countries have a strategy but no enforcement mechanism |
| Surprise Hawk | Brazil — 13/16, ahead of the US (4/16) and UK (5/16) |

*Data collected April 2026. Scoring reflects public disclosure and legislative status — 
not enforcement quality or outcomes.*